# 05 · Sampling Methods
### Practical LLM Engineering — Module 01: Fundamentals

> **What you'll learn**
> - How raw logits become the next token — every decoding strategy from scratch
> - Greedy, temperature, top-k, top-p (nucleus), min-p, and typical sampling
> - How repetition and frequency penalties work
> - Beam search and why it is rarely used for LLMs today
> - How to choose the right strategy for your use case
> - Engineering insights: speed, diversity, and quality tradeoffs

---


## 1. Overview

At every generation step the model produces a **logit vector** $\mathbf{z} \in \mathbb{R}^{|\mathcal{V}|}$ — one raw score per vocabulary token.

The **decoding strategy** is the algorithm that turns $\mathbf{z}$ into the next token $x_t$.

```
Transformer forward pass
        ↓
   logits  z ∈ ℝ^V
        ↓
   ┌─────────────────────────────┐
   │   Decoding strategy         │
   │   - temperature scaling     │
   │   - top-k / top-p / min-p   │
   │   - repetition penalty      │
   │   - beam search             │
   └─────────────────────────────┘
        ↓
   next token  x_t
```

Choosing the right strategy is a critical engineering decision — it directly controls:
- **Quality**: does the output make sense?
- **Diversity**: does the model repeat itself?
- **Creativity**: can the model produce surprising, useful outputs?
- **Speed**: beam search is much slower than sampling


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install torch transformers matplotlib numpy -q

import math
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import Counter

torch.manual_seed(42)
np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Libraries ready")
print(f"   Device : {device}")


In [ ]:
# Load GPT-2 (runs on CPU — no GPU needed)
print("Loading GPT-2...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model     = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
model.eval()

print(f"Vocab size       : {tokenizer.vocab_size:,}")
print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")

# Helper: get logits for the last token of a prompt
def get_logits(prompt: str) -> torch.Tensor:
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        return model(ids).logits[0, -1, :]   # shape: (vocab_size,)

# Helper: show top-k token probabilities as a bar chart
def plot_top_k(probs: torch.Tensor, k: int = 15, title: str = ""):
    top_probs, top_ids = probs.topk(k)
    top_probs = top_probs.cpu().numpy()
    top_toks  = [repr(tokenizer.decode([i])) for i in top_ids.cpu().tolist()]

    fig, ax = plt.subplots(figsize=(10, 3.5))
    bars = ax.bar(range(k), top_probs, color="#3498db", edgecolor="white", linewidth=0.5)
    ax.set_xticks(range(k))
    ax.set_xticklabels(top_toks, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Probability")
    ax.set_title(title or f"Top-{k} token probabilities")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

logits_demo = get_logits("The capital of France is")
probs_demo  = F.softmax(logits_demo, dim=-1)
plot_top_k(probs_demo, k=15, title='Top-15 tokens after "The capital of France is"')


## 3. Greedy Decoding

The simplest strategy: always pick the **most probable** token.

$$
x_t = \underset{v \in \mathcal{V}}{\arg\max}\; P(v \mid x_{<t})
$$

**Pros:**
- Deterministic and reproducible
- Fast (no sampling overhead)
- Good for factual tasks (QA, translation)

**Cons:**
- Repetitive — once the model finds a high-probability loop, it stays there
- Not creative — always the "safe" choice
- Does not maximise sequence probability (a greedy sequence can be globally suboptimal)

**When to use:** factual retrieval, structured output, code completion with a clear correct answer.


In [ ]:
def greedy_decode(logits: torch.Tensor) -> int:
    """Return the token id with the highest logit."""
    return logits.argmax().item()


# ── Demo ──────────────────────────────────────────────────────────────
prompt = "The capital of France is"
logits = get_logits(prompt)

greedy_id  = greedy_decode(logits)
greedy_tok = tokenizer.decode([greedy_id])
print(f"Prompt : {prompt!r}")
print(f"Greedy next token : {greedy_tok!r}  (id={greedy_id})")
print()

# ── Generate a full sequence greedily ────────────────────────────────
def generate_greedy(prompt: str, max_new: int = 40) -> str:
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new,
                             do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for prompt in [
    "The capital of France is",
    "Artificial intelligence will",
    "Once upon a time",
]:
    print(f"  {prompt!r}")
    print(f"  → {generate_greedy(prompt, 20)!r}")
    print()


## 4. Temperature Sampling

Temperature $\tau$ rescales the logits before softmax:

$$
P_\tau(v) = \frac{\exp(z_v / \tau)}{\sum_{v'} \exp(z_{v'} / \tau)}
$$

Then sample $x_t \sim P_\tau$.

**Effect of temperature:**

| $\tau$ | Distribution | Behaviour |
|---|---|---|
| $\tau \to 0$ | Spike on argmax | Equivalent to greedy |
| $\tau = 1$ | Model's raw distribution | Standard sampling |
| $\tau > 1$ | Flatter (more uniform) | More random / creative |
| $\tau \gg 1$ | Nearly uniform | Incoherent |

**Intuition:** dividing logits by $\tau < 1$ amplifies differences — the gap between the top token and the rest grows, making the distribution sharper.


In [ ]:
def temperature_sample(logits: torch.Tensor, temperature: float) -> int:
    """Sample from the temperature-scaled distribution."""
    if temperature <= 0:
        return logits.argmax().item()
    scaled = logits / temperature
    probs  = F.softmax(scaled, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# ── Visualise temperature effect ──────────────────────────────────────
logits = get_logits("The future of artificial intelligence is")
temps  = [0.1, 0.5, 1.0, 1.5, 2.0]
k_show = 12

fig, axes = plt.subplots(1, len(temps), figsize=(16, 3.5), sharey=False)
fig.suptitle("Effect of temperature on token probability distribution", fontsize=12)

for ax, tau in zip(axes, temps):
    probs = F.softmax(logits / tau, dim=-1)
    top_p, top_ids = probs.topk(k_show)
    top_p   = top_p.cpu().detach().numpy()
    labels  = [repr(tokenizer.decode([i])) for i in top_ids.cpu().tolist()]

    color = "#e74c3c" if tau < 0.5 else ("#3498db" if tau == 1.0 else "#2ecc71")
    ax.bar(range(k_show), top_p, color=color, edgecolor="white", linewidth=0.4)
    ax.set_xticks(range(k_show))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_title(f"τ = {tau}", fontsize=11)
    ax.set_xlabel("Token", fontsize=8)
    if ax == axes[0]: ax.set_ylabel("Probability")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Entropy as a function of temperature ─────────────────────────────
def entropy(probs: torch.Tensor) -> float:
    p = probs.clamp(min=1e-12)
    return -(p * p.log()).sum().item()

logits  = get_logits("The future of artificial intelligence is")
tau_range = np.linspace(0.1, 3.0, 100)
entropies = []
for tau in tau_range:
    p = F.softmax(logits / tau, dim=-1)
    entropies.append(entropy(p))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(tau_range, entropies, color="#9b59b6", lw=2)
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.6, label="τ=1 (model distribution)")
ax.set_xlabel("Temperature τ")
ax.set_ylabel("Entropy H(P) (nats)")
ax.set_title("Distribution entropy vs temperature")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Top-k Sampling

Top-k sampling restricts sampling to the **$k$ most probable tokens**, zeroing out the rest:

$$
P_k(v) = \begin{cases}
\dfrac{P(v)}{\sum_{u \in \mathcal{T}_k} P(u)} & \text{if } v \in \mathcal{T}_k \\
0 & \text{otherwise}
\end{cases}
$$

where $\mathcal{T}_k$ is the set of the top-$k$ tokens by probability.

**Pros:**
- Prevents sampling from the long tail of unlikely tokens
- Simple, fast, one hyperparameter

**Cons:**
- $k$ is context-independent — sometimes the top-$k$ is too narrow (peaked distribution), sometimes too wide (flat distribution)
- A fixed $k=50$ can include very low-probability tokens when the distribution is flat


In [ ]:
def top_k_sample(logits: torch.Tensor, k: int, temperature: float = 1.0) -> int:
    """Sample from the top-k tokens only."""
    logits = logits / max(temperature, 1e-8)

    # Zero out all logits below the k-th largest
    top_k_vals, _ = logits.topk(k)
    threshold      = top_k_vals[-1]
    filtered       = logits.masked_fill(logits < threshold, float("-inf"))

    probs = F.softmax(filtered, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# ── Visualise top-k truncation ────────────────────────────────────────
logits = get_logits("The most important thing in life is")
probs  = F.softmax(logits, dim=-1)
sorted_probs, sorted_ids = probs.sort(descending=True)
sorted_probs = sorted_probs[:50].cpu().detach().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
fig.suptitle("Top-k sampling: truncating the vocabulary", fontsize=12)

for ax, k in zip(axes, [5, 20, 50]):
    colors = ["#3498db" if i < k else "#ecf0f1" for i in range(50)]
    ax.bar(range(50), sorted_probs, color=colors, edgecolor="none")
    ax.axvline(k - 0.5, color="#e74c3c", linestyle="--", lw=1.5, label=f"k={k} cutoff")
    ax.set_title(f"Top-k={k}")
    ax.set_xlabel("Token rank")
    ax.set_ylabel("Probability" if ax == axes[0] else "")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Top-p (Nucleus) Sampling

Top-p sampling (Holtzman et al., 2020) is **adaptive**: it selects the **smallest set of tokens** whose cumulative probability exceeds threshold $p$:

$$
\mathcal{N}_p = \underset{\mathcal{S} \subseteq \mathcal{V}}{\arg\min} \left\{ |\mathcal{S}| \;\Big|\; \sum_{v \in \mathcal{S}} P(v) \geq p \right\}
$$

where tokens in $\mathcal{S}$ are sorted by descending probability.

**Adaptive behaviour:**
- When the distribution is **peaked** (confident model): $\mathcal{N}_p$ is small — only the top few tokens
- When the distribution is **flat** (uncertain model): $\mathcal{N}_p$ is large — many tokens qualify

This makes top-p much more robust than fixed top-k across different contexts.

**Common defaults:** $p = 0.9$ or $p = 0.95$, $\tau = 0.8$–$1.0$


In [ ]:
def top_p_sample(logits: torch.Tensor, p: float, temperature: float = 1.0) -> int:
    """Nucleus (top-p) sampling."""
    logits = logits / max(temperature, 1e-8)
    probs  = F.softmax(logits, dim=-1)

    # Sort descending and compute cumulative probabilities
    sorted_probs, sorted_indices = probs.sort(descending=True)
    cumulative_probs = sorted_probs.cumsum(dim=-1)

    # Remove tokens once cumulative probability exceeds p
    # Shift right by 1 so we keep the token that first crosses p
    remove_mask = cumulative_probs - sorted_probs > p
    sorted_probs[remove_mask] = 0.0

    # Renormalise
    sorted_probs /= sorted_probs.sum()

    # Sample and map back to original vocab indices
    sampled_rank = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_indices[sampled_rank].item()


# ── Visualise nucleus size across different prompts ───────────────────
test_prompts = [
    ("Confident model: 'Paris is the capital of'",
     "Paris is the capital of"),
    ("Uncertain model: 'The meaning of life is'",
     "The meaning of life is"),
    ("Very uncertain: 'The next word could be anything:'",
     "The next word could be anything:"),
]

p_threshold = 0.9

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Nucleus size adapts to distribution shape  (p={p_threshold})", fontsize=12)

for ax, (title, prompt) in zip(axes, test_prompts):
    logits = get_logits(prompt)
    probs  = F.softmax(logits, dim=-1)
    sorted_probs, _ = probs.sort(descending=True)
    cumprobs = sorted_probs.cumsum(dim=-1).cpu().detach().numpy()
    sorted_p = sorted_probs.cpu().detach().numpy()

    nucleus_size = int((cumprobs < p_threshold).sum()) + 1
    colors = ["#3498db" if i < nucleus_size else "#ecf0f1" for i in range(60)]

    ax.bar(range(60), sorted_p[:60], color=colors, edgecolor="none")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("Token rank")
    ax.set_ylabel("Probability" if ax == axes[0] else "")
    ax.set_title(f"{title.split(':')[0]}
nucleus size = {nucleus_size}", fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    # Mark nucleus boundary
    ax.axvline(nucleus_size - 0.5, color="#e74c3c", linestyle="--", lw=1.5)

plt.tight_layout()
plt.show()


## 7. Min-p Sampling

Min-p sampling (Nguyen et al., 2024) offers a simpler and often better alternative to top-p.

Instead of a cumulative probability threshold, it sets a **minimum probability floor** relative to the top token:

$$
\mathcal{M}_p = \{ v \in \mathcal{V} \mid P(v) \geq p \cdot P(v^*) \}
$$

where $v^* = \arg\max_v P(v)$ is the top token and $p \in (0, 1)$ is the min-p threshold.

**Intuition:** a token must be at least a fraction $p$ as probable as the best token to be considered.

**Advantages over top-p:**
- Never excludes the top token (top-p can fail with extreme distributions)
- Scales naturally with the peak probability — more conservative when the model is confident
- Tends to produce better diversity-quality tradeoffs in practice

**Common default:** $p = 0.05$–$0.1$


In [ ]:
def min_p_sample(logits: torch.Tensor, p: float, temperature: float = 1.0) -> int:
    """Min-p sampling: keep tokens with prob >= p * max_prob."""
    logits = logits / max(temperature, 1e-8)
    probs  = F.softmax(logits, dim=-1)

    # Compute the threshold relative to the top token
    max_prob  = probs.max()
    threshold = p * max_prob

    # Zero out tokens below threshold
    filtered = probs.masked_fill(probs < threshold, 0.0)
    filtered /= filtered.sum()

    return torch.multinomial(filtered, num_samples=1).item()


# ── Compare nucleus sizes: top-p vs min-p ────────────────────────────
prompts_compare = [
    "Paris is the capital of",
    "The meaning of life is",
    "I went to the",
]

print(f"{'Prompt':<35} {'top-p(0.9) size':>16} {'min-p(0.05) size':>17}")
print("-" * 72)
for prompt in prompts_compare:
    logits = get_logits(prompt)
    probs  = F.softmax(logits, dim=-1)

    # top-p nucleus size
    sorted_p, _ = probs.sort(descending=True)
    cumprobs = sorted_p.cumsum(dim=-1).cpu().detach().numpy()
    top_p_size = int((cumprobs < 0.9).sum()) + 1

    # min-p nucleus size
    max_p     = probs.max().item()
    threshold = 0.05 * max_p
    min_p_size = int((probs >= threshold).sum().item())

    print(f"{prompt:<35} {top_p_size:>16} {min_p_size:>17}")


## 8. Typical Sampling

Typical sampling (Meister et al., 2023) is motivated by information theory.

Instead of keeping the most *probable* tokens, it keeps tokens whose **surprisal** $-\log P(v)$ is closest to the **conditional entropy** $H$ of the distribution:

$$
\mathcal{T}_\tau = \left\{ v \in \mathcal{V} \;\Big|\; \Big| {-\log P(v) - H(P)} \Big| \leq \delta \right\}
$$

where:
$$
H(P) = -\sum_{v} P(v) \log P(v)
$$

**Intuition:** humans tend to produce tokens that are "typically surprising" — not the most predictable, not the most shocking. Typical sampling mimics this by staying near the expected information content.

**Practical note:** top-p and min-p are more widely used; typical sampling is useful to know as an alternative with strong theoretical grounding.


In [ ]:
def typical_sample(logits: torch.Tensor, mass: float = 0.9,
                   temperature: float = 1.0) -> int:
    """Typical sampling — keeps tokens near the distribution's entropy."""
    logits = logits / max(temperature, 1e-8)
    probs  = F.softmax(logits, dim=-1)

    # Compute entropy of the distribution
    log_probs = F.log_softmax(logits, dim=-1)
    H = -(probs * log_probs).sum().item()

    # Surprisal: -log P(v)
    surprisal = -log_probs   # shape: (vocab,)

    # Distance from entropy
    dist_from_entropy = (surprisal - H).abs()

    # Sort by distance, keep smallest until mass is covered
    sorted_dist, sorted_idx = dist_from_entropy.sort()
    sorted_probs_by_dist    = probs[sorted_idx]
    cumsum = sorted_probs_by_dist.cumsum(dim=-1)

    # Mask: keep tokens until cumulative probability >= mass
    keep_mask                     = torch.zeros_like(probs, dtype=torch.bool)
    keep_mask[sorted_idx[cumsum - sorted_probs_by_dist < mass]] = True

    filtered       = probs.clone()
    filtered[~keep_mask] = 0.0
    filtered      /= filtered.sum()

    return torch.multinomial(filtered, num_samples=1).item()


# ── Demo: compare all sampling methods on the same prompt ────────────
prompt  = "The best way to learn programming is"
logits  = get_logits(prompt)
n_draws = 20

results = {}
for name, fn in [
    ("Greedy",        lambda: greedy_decode(logits)),
    ("Temp τ=0.8",    lambda: temperature_sample(logits, 0.8)),
    ("Top-k k=50",    lambda: top_k_sample(logits, 50, 0.8)),
    ("Top-p p=0.9",   lambda: top_p_sample(logits, 0.9, 0.8)),
    ("Min-p p=0.05",  lambda: min_p_sample(logits, 0.05, 0.8)),
    ("Typical τ=0.9", lambda: typical_sample(logits, 0.9, 0.8)),
]:
    draws = [fn() for _ in range(n_draws)]
    tokens = [tokenizer.decode([d]) for d in draws]
    results[name] = Counter(tokens)

print(f'Next-token distribution for: "{prompt}"')
print(f"(top-3 tokens from {n_draws} draws each)\n")
for method, counter in results.items():
    top3 = counter.most_common(3)
    top3_str = "  ".join(f"{t!r}:{c}" for t, c in top3)
    print(f"  {method:<18} → {top3_str}")


## 9. Repetition and Frequency Penalties

Without penalties, greedy and low-temperature sampling often fall into repetition loops:

```
"The cat sat on the mat. The cat sat on the mat. The cat sat on the mat..."
```

Two standard penalty mechanisms address this.

---

### 9.1 Repetition Penalty (Keskar et al., 2019)

Divides the logit of any token that has already appeared in the context:

$$
z_v' = \begin{cases}
z_v / \alpha & \text{if } v \in \mathcal{C} \text{ and } z_v > 0 \\
z_v \times \alpha & \text{if } v \in \mathcal{C} \text{ and } z_v < 0
\end{cases}
$$

where $\mathcal{C}$ is the set of tokens seen so far and $\alpha > 1$ is the penalty strength.

---

### 9.2 Frequency Penalty (OpenAI-style)

Subtracts a penalty proportional to how many times a token has already appeared:

$$
z_v' = z_v - \beta \cdot \text{count}(v, \mathcal{C})
$$

A related **presence penalty** applies a flat subtraction for any token that has appeared at all.


In [ ]:
def apply_repetition_penalty(logits: torch.Tensor,
                              input_ids: list[int],
                              penalty: float = 1.3) -> torch.Tensor:
    """Reduce logits for tokens already in the context."""
    logits = logits.clone()
    for token_id in set(input_ids):
        if logits[token_id] > 0:
            logits[token_id] /= penalty
        else:
            logits[token_id] *= penalty
    return logits


def apply_frequency_penalty(logits: torch.Tensor,
                              input_ids: list[int],
                              freq_penalty: float = 0.5) -> torch.Tensor:
    """Subtract penalty proportional to token frequency in context."""
    logits  = logits.clone()
    counts  = Counter(input_ids)
    for token_id, count in counts.items():
        logits[token_id] -= freq_penalty * count
    return logits


# ── Show effect on logits ─────────────────────────────────────────────
context = "The cat sat on the mat the cat sat on"
ctx_ids = tokenizer.encode(context)
logits  = get_logits(context)
probs_orig = F.softmax(logits, dim=-1)

logits_rep  = apply_repetition_penalty(logits, ctx_ids, penalty=1.5)
logits_freq = apply_frequency_penalty(logits, ctx_ids, freq_penalty=0.5)
probs_rep   = F.softmax(logits_rep, dim=-1)
probs_freq  = F.softmax(logits_freq, dim=-1)

# Focus on a few tokens that appear in context
watch_tokens = ["the", " the", " cat", " sat", " on", " mat", " a", " an"]
watch_ids    = [tokenizer.encode(t)[0] for t in watch_tokens]

print(f"{'Token':<12} {'P(orig)':>10} {'P(rep×1.5)':>12} {'P(freq-0.5)':>13}")
print("-" * 50)
for tok, tid in zip(watch_tokens, watch_ids):
    p0 = probs_orig[tid].item()
    pr = probs_rep[tid].item()
    pf = probs_freq[tid].item()
    print(f"{tok!r:<12} {p0:>10.4f} {pr:>12.4f} {pf:>13.4f}")


In [ ]:
# ── Demonstrate: greedy without vs with repetition penalty ──────────
def generate_with_penalty(prompt: str, max_new: int = 60,
                           rep_penalty: float = 1.0, device: str = "cpu") -> str:
    ids      = tokenizer.encode(prompt, return_tensors="pt").to(device)
    past     = None
    current  = ids.clone()
    generated_ids = ids[0].tolist()

    with torch.no_grad():
        for _ in range(max_new):
            out    = model(current, past_key_values=past, use_cache=True)
            logits = out.logits[0, -1, :]
            past   = out.past_key_values

            if rep_penalty > 1.0:
                logits = apply_repetition_penalty(logits, generated_ids, rep_penalty)

            next_id = logits.argmax().item()
            generated_ids.append(next_id)
            current = torch.tensor([[next_id]], device=device)

            if next_id == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated_ids, skip_special_tokens=True)


prompt = "The cat sat on the mat and the cat"
print("Without repetition penalty:")
print(generate_with_penalty(prompt, max_new=50, rep_penalty=1.0))
print()
print("With repetition penalty (1.3):")
print(generate_with_penalty(prompt, max_new=50, rep_penalty=1.3))


## 10. Beam Search

All methods above are **single-path** — they commit to one token at each step.

**Beam search** maintains $B$ candidate sequences (beams) simultaneously, expanding the most probable beams at each step.

```
Step 0:   [<bos>]

Step 1 (B=3 beams):
  Beam 1:  [<bos>, "The"]         log P = -0.5
  Beam 2:  [<bos>, "A"]           log P = -0.9
  Beam 3:  [<bos>, "In"]          log P = -1.1

Step 2:
  Beam 1:  [<bos>, "The", "cat"]  log P = -1.2
  Beam 2:  [<bos>, "The", "dog"]  log P = -1.4
  Beam 3:  [<bos>, "A", "cat"]    log P = -1.6
  ...
```

The score for beam $b$ after step $t$ is the **sum of log-probabilities**:

$$
\text{score}(b_t) = \sum_{i=1}^{t} \log P(x_i \mid x_{<i})
$$

**Length penalty** prevents short sequences from dominating:

$$
\text{score}_{\text{norm}}(b) = \frac{\text{score}(b)}{|b|^\alpha}
$$

**Why beam search is rarely used for LLMs today:**
- Compute cost scales as $\mathcal{O}(B \cdot T \cdot V)$ — $B$× more expensive
- Produces **boring, generic** text (maximises likelihood ≠ best text)
- Still used for: machine translation, summarisation, structured output


In [ ]:
def beam_search(model, tokenizer, prompt: str,
                num_beams: int = 3, max_new: int = 20,
                length_penalty: float = 1.0, device: str = "cpu") -> list[dict]:
    """
    Simple beam search implementation.
    Returns top beams sorted by normalised score.
    """
    model.eval()
    eos_id = tokenizer.eos_token_id

    # Initialise beams: each is (token_ids, log_prob_sum, done)
    init_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    beams = [(init_ids[0].tolist(), 0.0, False)]

    with torch.no_grad():
        for step in range(max_new):
            if all(b[2] for b in beams):
                break   # all beams finished

            candidates = []
            for seq, score, done in beams:
                if done:
                    candidates.append((seq, score, True))
                    continue

                ids    = torch.tensor([seq], device=device)
                logits = model(ids).logits[0, -1, :]
                log_p  = F.log_softmax(logits, dim=-1)

                # Expand top num_beams tokens
                top_log_p, top_ids = log_p.topk(num_beams)
                for lp, tid in zip(top_log_p.tolist(), top_ids.tolist()):
                    new_seq   = seq + [tid]
                    new_score = score + lp
                    is_done   = (tid == eos_id)
                    candidates.append((new_seq, new_score, is_done))

            # Keep top num_beams candidates by normalised score
            candidates.sort(
                key=lambda x: x[1] / (len(x[0]) ** length_penalty),
                reverse=True
            )
            beams = candidates[:num_beams]

    results = []
    for seq, score, done in beams:
        norm_score = score / (len(seq) ** length_penalty)
        text       = tokenizer.decode(seq, skip_special_tokens=True)
        results.append({"text": text, "score": norm_score, "length": len(seq)})
    return results


# ── Run beam search ──────────────────────────────────────────────────
prompt  = "The best programming language for machine learning is"
results = beam_search(model, tokenizer, prompt, num_beams=3,
                       max_new=15, device=device)

print(f'Beam search results for: "{prompt}"\n')
for i, r in enumerate(results):
    print(f"  Beam {i+1}  score={r['score']:.3f}  len={r['length']}")
    print(f"  {r['text']!r}")
    print()


## 11. Side-by-Side Comparison

In [ ]:
# ── Generate 3 samples from each method ──────────────────────────────
prompt = "In the year 2050, artificial intelligence"

strategies = {
    "Greedy":           dict(do_sample=False),
    "Temp τ=0.7":       dict(do_sample=True, temperature=0.7),
    "Top-k k=50":       dict(do_sample=True, top_k=50, temperature=0.8),
    "Top-p p=0.9":      dict(do_sample=True, top_p=0.9, temperature=0.8),
    "Beam B=4":         dict(do_sample=False, num_beams=4),
    "Temp τ=1.5":       dict(do_sample=True, temperature=1.5),
}

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
print(f'Prompt: "{prompt}"\n{"="*60}\n')

for name, kwargs in strategies.items():
    torch.manual_seed(42)
    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=30,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    continuation = text[len(prompt):]
    print(f"[{name}]")
    print(f"  ...{continuation}")
    print()


## 12. Engineering Guide: Choosing a Decoding Strategy

### Quick decision table

| Use case | Recommended strategy | Typical settings |
|---|---|---|
| Factual QA, structured output | Greedy or low temperature | `temperature=0.0–0.3` |
| Code generation | Greedy or low temp + rep penalty | `temperature=0.2, rep=1.1` |
| Chat / instruction following | Top-p + temperature | `top_p=0.9, temp=0.7` |
| Creative writing | Top-p or min-p + high temp | `top_p=0.95, temp=1.0–1.2` |
| Summarisation | Beam search or low temp | `num_beams=4` |
| Translation | Beam search | `num_beams=5, length_penalty=0.6` |
| Diverse outputs (e.g. brainstorm) | Top-p + high temp | `top_p=0.95, temp=1.2` |

---

### Speed comparison

| Strategy | Relative speed | Notes |
|---|---|---|
| Greedy | 1× (baseline) | Single forward pass |
| Temperature | 1× | Same as greedy + softmax sample |
| Top-k | ~1× | Small overhead for topk sort |
| Top-p | ~1× | Small overhead for cumsum |
| Min-p | ~1× | Same as top-p |
| Beam (B=4) | ~4× slower | $B$ forward passes per step |
| Beam (B=8) | ~8× slower | Memory also ×8 |


In [ ]:
# ── Timing: greedy vs top-p vs beam ──────────────────────────────────
import time

input_ids = tokenizer.encode(
    "The most important development in AI research",
    return_tensors="pt"
).to(device)

timings = {}
for name, kwargs in [
    ("Greedy",    dict(do_sample=False)),
    ("Top-p",     dict(do_sample=True, top_p=0.9, temperature=0.8)),
    ("Beam B=2",  dict(do_sample=False, num_beams=2)),
    ("Beam B=4",  dict(do_sample=False, num_beams=4)),
]:
    times_i = []
    for _ in range(3):   # average over 3 runs
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(input_ids, max_new_tokens=30,
                           pad_token_id=tokenizer.eos_token_id, **kwargs)
        times_i.append(time.perf_counter() - t0)
    timings[name] = np.mean(times_i)

baseline = timings["Greedy"]
print(f"{'Strategy':<14} {'Time (s)':>10} {'Relative':>10}")
print("-" * 36)
for name, t in timings.items():
    print(f"{name:<14} {t:>10.3f} {t/baseline:>10.2f}×")


## 13. Exercises

1. **Diversity metric:** Generate 20 continuations of the same prompt with greedy, top-p (p=0.9), and temperature (τ=1.2). Compute the **type-token ratio** (unique tokens / total tokens) for each. Which strategy produces the most diverse outputs?

2. **Temperature calibration:** For a factual prompt ("The year the Eiffel Tower was built is"), sweep temperature from 0.1 to 2.0 in steps of 0.1. Plot the probability of the correct answer token vs temperature. At what temperature does the model "forget" the right answer?

3. **Top-p sensitivity:** For 10 different prompts, record the nucleus size at p=0.9. Is there a correlation between nucleus size and the entropy of the distribution? Plot nucleus size vs entropy.

4. **Beam search diversity:** Run beam search with `num_beams=5` and `num_return_sequences=5`. How similar are the 5 beams to each other? Compute pairwise BLEU or token overlap between them.

5. **Combined penalties:** Implement a sampler that combines top-p, temperature, and frequency penalty in the correct order (temperature → frequency penalty → top-p → sample). Verify the order matters by swapping penalty and top-p.

---

## 14. Key Takeaways

| Strategy | Core idea | Best for |
|---|---|---|
| **Greedy** | $\arg\max$ at every step | Deterministic, factual |
| **Temperature** | Rescale logits; control entropy | General-purpose tuning knob |
| **Top-k** | Sample from top-$k$ tokens only | Simple, fast truncation |
| **Top-p** | Adaptive nucleus by cumulative prob | Most widely used default |
| **Min-p** | Floor relative to max prob | Strong alternative to top-p |
| **Typical** | Sample near distribution entropy | Theoretically motivated |
| **Rep penalty** | Penalise already-seen tokens | Prevent repetition loops |
| **Beam search** | Multi-path, maximise sequence prob | Translation, summarisation |

**Default recipe for most LLM chat applications:**
`temperature=0.7, top_p=0.9, repetition_penalty=1.1`

---

**Next notebook →** `02_prompting/01_zero_shot_prompting.ipynb` — Module 01 complete! 🎉
